Simple GENAI app using langchain


In [1]:
# data ingestion
from langchain_community.document_loaders import WebBaseLoader
# internally webbaseloader uses beautifulsoup4 and requests.
loader = WebBaseLoader("https://suquamish.nsn.us/home/about-us/chief-seattle-speech/")
docs = loader.load()
docs

c:\Generative AI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


[Document(metadata={'source': 'https://suquamish.nsn.us/home/about-us/chief-seattle-speech/', 'title': 'Chief Seattle Speech | The Suquamish Tribe', 'language': 'en-US'}, page_content='\n\n\n\n\n\n\n\n\n\nChief Seattle Speech | The Suquamish Tribe\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n \n\n\n\n\n\n\n\n\nFacebook \n\n\nHOME\nABOUT\n\nPeople of the Clear Salt Water\nHistory & Culture\nSuquamish Today\nFrequently Asked Questions\nChief Seattle Days\n\n\nGOVERNMENT\n\nTribal Council\nBoards & Committees\nEnrollment\nGaming Commission\nCommunity Investment Report\nGrants\nSuquamish Tribal Code\n\n\nDEPARTMENTS\n\nAll Departments\nCommunity Development\nCultural Resources\nEducation\nEmergency Management\nFamily & Friends Center\nFinance\nFisheries\nHealth Division\nHuman Services\nNatural Resources\nMuseum\nPolice & Courts\nWellness Center\n\n\nSUQUAMISH FOUNDATION\n\nAbout the Foundation\nBoard of Trustees\nGet Involved\nGrant Programs\nShellf

In [2]:
# data transformation
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)
texts = text_splitter.split_documents(docs)
texts

[Document(metadata={'source': 'https://suquamish.nsn.us/home/about-us/chief-seattle-speech/', 'title': 'Chief Seattle Speech | The Suquamish Tribe', 'language': 'en-US'}, page_content='Chief Seattle Speech | The Suquamish Tribe\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n \n\n\n\n\n\n\n\n\nFacebook \n\n\nHOME\nABOUT\n\nPeople of the Clear Salt Water\nHistory & Culture\nSuquamish Today\nFrequently Asked Questions\nChief Seattle Days\n\n\nGOVERNMENT\n\nTribal Council\nBoards & Committees\nEnrollment\nGaming Commission\nCommunity Investment Report\nGrants\nSuquamish Tribal Code\n\n\nDEPARTMENTS'),
 Document(metadata={'source': 'https://suquamish.nsn.us/home/about-us/chief-seattle-speech/', 'title': 'Chief Seattle Speech | The Suquamish Tribe', 'language': 'en-US'}, page_content='DEPARTMENTS\n\nAll Departments\nCommunity Development\nCultural Resources\nEducation\nEmergency Management\nFamily & Friends Center\nFinance\nFisheries\nHealth Division\nHu

In [3]:
# data embedding
from langchain_community.embeddings import OllamaEmbeddings
embeddings = OllamaEmbeddings(model="snowflake-arctic-embed")


C:\Users\Rajeev Pandey\AppData\Local\Temp\ipykernel_22264\1616023694.py:3: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model="snowflake-arctic-embed")


In [4]:
#  storing the embeddings in a vector database
from langchain_community.vectorstores import FAISS
db = FAISS.from_documents(texts,embeddings)



In [5]:
# query from vector store db
query ="Who is chief Seattle talking to?"
result = db.similarity_search(query)
result[0].page_content


'Chief Seattle Speech | The Suquamish Tribe\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n \n\n\n\n\n\n\n\n\nFacebook \n\n\nHOME\nABOUT\n\nPeople of the Clear Salt Water\nHistory & Culture\nSuquamish Today\nFrequently Asked Questions\nChief Seattle Days\n\n\nGOVERNMENT\n\nTribal Council\nBoards & Committees\nEnrollment\nGaming Commission\nCommunity Investment Report\nGrants\nSuquamish Tribal Code\n\n\nDEPARTMENTS'

In [14]:
# Retriever
# when we want to provide context to LLMs for Q&A we use retrievers
# LLM (Ollama)
from langchain_community.chat_models import ChatOllama

# Prompts
from langchain_core.prompts import ChatPromptTemplate

# Old chain helpers now live here in v1.x
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain


In [16]:
# 1. Retriever
retriever = db.as_retriever()

# 2. LLM from Ollama
llm = ChatOllama(
    model="gemma3:1b",   # or whatever you pulled
    temperature=0.2,
)

In [ ]:


# 3. Prompt
prompt = ChatPromptTemplate.from_template(
    """
You are an assistant answering questions about Chief Seattle's speech.
Use ONLY the context provided. If you don't know, say you don't know.

Context:
{context}

Question:
{input}

Answer:
"""
)




Chief Seattle is talking to the Great Spirit.


In [23]:
# 4. Chain that stuffs docs into the prompt
# we create this chain bceause the retrival chain needs it becasue it combines the docs retrieved from the retriever into the prompt
doc_chain = create_stuff_documents_chain(llm, prompt)



In [25]:
# we convert the vectorstore to a retriever object
retriever = db.as_retriever()
from langchain_classic.chains import create_retrieval_chain
retrieval_chain=create_retrieval_chain(retriever, doc_chain)

In [27]:
# get response from llm
response=retrieval_chain.invoke({"input": "Who is Chief Seattle talking to?"})

In [28]:
response

{'input': 'Who is Chief Seattle talking to?',
 'context': [Document(id='aaa8d9ea-1892-41e9-86ce-66bc1282a58e', metadata={'source': 'https://suquamish.nsn.us/home/about-us/chief-seattle-speech/', 'title': 'Chief Seattle Speech | The Suquamish Tribe', 'language': 'en-US'}, page_content='Chief Seattle Speech | The Suquamish Tribe\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n \n\n\n\n\n\n\n\n\nFacebook \n\n\nHOME\nABOUT\n\nPeople of the Clear Salt Water\nHistory & Culture\nSuquamish Today\nFrequently Asked Questions\nChief Seattle Days\n\n\nGOVERNMENT\n\nTribal Council\nBoards & Committees\nEnrollment\nGaming Commission\nCommunity Investment Report\nGrants\nSuquamish Tribal Code\n\n\nDEPARTMENTS'),
  Document(id='7bb3ea2b-b9c0-4475-8d9f-9265ba7eb0b8', metadata={'source': 'https://suquamish.nsn.us/home/about-us/chief-seattle-speech/', 'title': 'Chief Seattle Speech | The Suquamish Tribe', 'language': 'en-US'}, page_content='CALENDAR\nCONTACT US\n\nPho

In [29]:
response["context"]

[Document(id='aaa8d9ea-1892-41e9-86ce-66bc1282a58e', metadata={'source': 'https://suquamish.nsn.us/home/about-us/chief-seattle-speech/', 'title': 'Chief Seattle Speech | The Suquamish Tribe', 'language': 'en-US'}, page_content='Chief Seattle Speech | The Suquamish Tribe\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n \n\n\n\n\n\n\n\n\nFacebook \n\n\nHOME\nABOUT\n\nPeople of the Clear Salt Water\nHistory & Culture\nSuquamish Today\nFrequently Asked Questions\nChief Seattle Days\n\n\nGOVERNMENT\n\nTribal Council\nBoards & Committees\nEnrollment\nGaming Commission\nCommunity Investment Report\nGrants\nSuquamish Tribal Code\n\n\nDEPARTMENTS'),
 Document(id='7bb3ea2b-b9c0-4475-8d9f-9265ba7eb0b8', metadata={'source': 'https://suquamish.nsn.us/home/about-us/chief-seattle-speech/', 'title': 'Chief Seattle Speech | The Suquamish Tribe', 'language': 'en-US'}, page_content='CALENDAR\nCONTACT US\n\nPhone/Address Directory\nCareers with Suquamish Tribe\nMedia C